In [15]:
import sim          
import sympy as sp  
import numpy as np
import time
import math

def connect(port):
    sim.simxFinish(-1)
    clientID=sim.simxStart('127.0.0.1',port,True,True,2000,5) # Conectarse
    if clientID == 0: print("conectado a", port)
    else: print("no se pudo conectar")
    return clientID

In [16]:
clientID = connect(19999)

retCode,sensorHandle=sim.simxGetObjectHandle(clientID,'Vision_sensor',sim.simx_opmode_blocking)
retCode, resolution, image=sim.simxGetVisionSensorImage(clientID,sensorHandle,0,sim.simx_opmode_oneshot_wait)

retCode,ruedaDerecha=sim.simxGetObjectHandle(clientID,'RuedaR',sim.simx_opmode_blocking)
retCode,ruedaIzquierda=sim.simxGetObjectHandle(clientID,'RuedaL',sim.simx_opmode_blocking)

retCode,suction=sim.simxGetObjectHandle(clientID,'suctionPad',sim.simx_opmode_blocking)

ret,ultrasonidoDerecha=sim.simxGetObjectHandle(clientID,'SensorR',sim.simx_opmode_blocking)
ret,ultrasonidoIzquierda=sim.simxGetObjectHandle(clientID,'SensorL',sim.simx_opmode_blocking)
ret,ultrasonidoDelante=sim.simxGetObjectHandle(clientID,'SensorD',sim.simx_opmode_blocking)
ret,ultrasonidoAtras=sim.simxGetObjectHandle(clientID,'SensorA',sim.simx_opmode_blocking)

conectado a 19999


In [17]:
def setEffector(val):
# function that triggers the end effector remotely
# val is Int with value 0 or 1 to disable or activate the final actuator.
    res,retInts,retFloats,retStrings,retBuffer=sim.simxCallScriptFunction(clientID,
        "suctionPad", sim.sim_scripttype_childscript,"setEffector",[val],[],[],"", sim.simx_opmode_blocking)
    return res

def obtenerDistanciaSensor(ultrasonido):
    errorCode, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector=sim.simxReadProximitySensor(clientID,ultrasonido, sim.simx_opmode_blocking)
    sensor_val=np.linalg.norm(detectedPoint)

    return sensor_val

In [18]:
def moverCasilla(v):
    tiempo = 6.5 / v
    print(tiempo)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, v, sim.simx_opmode_blocking)
    time.sleep(tiempo)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, 0, sim.simx_opmode_blocking)

def movimientoContinuo(velocidad, direccion):
    v = velocidad * direccion
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, v, sim.simx_opmode_blocking)
    

def giro90(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    tiempo = radianes_rueda / v
    print(tiempo)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   -v, sim.simx_opmode_blocking)
    
    time.sleep(tiempo)
    
    # 5. Frenamos ambos motores
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,    0, sim.simx_opmode_blocking)


def giroContinuo(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   -v, sim.simx_opmode_blocking)

def detener():
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)
                


In [21]:
detener()
derecha = obtenerDistanciaSensor(ultrasonidoDerecha)
izquierda = obtenerDistanciaSensor(ultrasonidoIzquierda)
delante = obtenerDistanciaSensor(ultrasonidoDelante)
atras = obtenerDistanciaSensor(ultrasonidoAtras)

print(derecha)
print(izquierda)

0.5430885966816646
0.3169231102103376


In [20]:
moverCasilla(0.5)
moverCasilla(0.5)
giro90(0.5,1)
#girarDerecha(2)

13.0
13.0
6.675884388878312
